# 5.24 Probabilistic programming

**Lesson tagline.** A probabilistic program writes the data-generating story; inference conditions that story on observations.

Probabilistic programming ties Bayesian inference, graphical models, MCMC, HMC, and VI into a modeling language. This notebook builds a tiny trace enumerator and compares approximate trace inference with exact conditioning. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 524
rng = np.random.default_rng(SEED)

def normalize(values):
    arr = np.asarray(values, dtype=float)
    total = arr.sum()
    if total <= 0:
        raise ValueError("zero likelihood: observations are impossible")
    return arr / total

def enumerate_traces(prior_probs, likelihoods, observation):
    rows = []
    evidence = 0.0
    for z, prior in enumerate(prior_probs):
        like = likelihoods[z][observation]
        joint = prior * like
        rows.append({"z": z, "prior": prior, "likelihood": like, "joint": joint})
        evidence += joint
    posterior = np.array([row["joint"] for row in rows]) / evidence
    return rows, evidence, posterior

def build_ppl_ladder():
    ladder = []
    ladder.append({"name": "D1 two-trace Bernoulli", "prior": np.array([0.7, 0.3]), "likelihood": np.array([[0.9, 0.1], [0.2, 0.8]]), "obs": 1})
    ladder.append({"name": "D2 four-trace latent program", "prior": np.array([0.1, 0.2, 0.4, 0.3]), "likelihood": np.array([[0.8, 0.2], [0.5, 0.5], [0.2, 0.8], [0.1, 0.9]]), "obs": 1})
    ladder.append({"name": "D3 bimodal branch program", "prior": np.array([0.45, 0.05, 0.05, 0.45]), "likelihood": np.array([[0.15, 0.85], [0.9, 0.1], [0.9, 0.1], [0.2, 0.8]]), "obs": 1})
    ladder.append({"name": "D4 correlated 2-D observation", "prior": np.array([0.2, 0.3, 0.3, 0.2]), "likelihood": np.array([[0.9, 0.1], [0.4, 0.6], [0.3, 0.7], [0.85, 0.15]]), "obs": 1})
    ladder.append({"name": "D5 impossible/bad geometry", "prior": np.array([0.05, 0.1, 0.15, 0.2, 0.2, 0.15, 0.1, 0.05]), "likelihood": np.array([[1.0, 0.0], [0.99, 0.01], [0.95, 0.05], [0.6, 0.4], [0.4, 0.6], [0.05, 0.95], [0.01, 0.99], [0.0, 1.0]]), "obs": 1})
    return ladder

def importance_trace_inference(rung, n, random_state):
    prior = rung["prior"]
    draws = random_state.choice(len(prior), size=n, p=prior)
    weights = rung["likelihood"][draws, rung["obs"]]
    mass = np.bincount(draws, weights=weights, minlength=len(prior))
    return normalize(mass)

def total_variation(p, q):
    return 0.5 * np.sum(np.abs(normalize(p) - normalize(q)))

## The concept, built once (D1)
A probabilistic program produces latent trace choices $z$ and observations $y$. Conditioning applies
$$p(z\mid y)\propto p(z)\prod_i p(y_i\mid z).$$
Enumeration makes that trace table explicit before using approximate inference on larger rungs.

In [ ]:
prior = np.array([0.7, 0.3])
likelihood = np.array([[0.9, 0.1], [0.2, 0.8]])
rows, evidence, posterior = enumerate_traces(prior, likelihood, 1)
assert abs(evidence - 0.31) < 1e-12
assert abs(posterior[1] - (0.24 / 0.31)) < 1e-12
assert abs(posterior[1] - 0.7741935483870968) < 1e-12
print("evidence", evidence)
print("posterior", posterior)

The trace joint table is prior times likelihood. Summing over every possible observation gives total probability $1$; posterior predictive probabilities reuse the same likelihood table under posterior trace weights.

In [ ]:
joint_table = prior[:, None] * likelihood
posterior_predictive = posterior @ likelihood
assert abs(joint_table.sum() - 1.0) < 1e-12
assert abs(posterior_predictive.sum() - 1.0) < 1e-12
print("joint table", joint_table)
print("posterior predictive", posterior_predictive)

## The dataset ladder
The F10 ladder is a sequence of tiny probabilistic programs: two traces, four traces, bimodal branches, correlated 2-D latent states, and a high-dimensional trace space with impossible or near-impossible observations.

In [ ]:
ladder = build_ppl_ladder()
for i, rung in enumerate(ladder, start=1):
    print(i, rung["name"], "traces=", len(rung["prior"]), "obs=", rung["obs"])

## Run the SAME method across D1-D5
Collect the plan metric on every rung from D1–D5 with the same algorithmic implementation, then print a compact per-rung table.

In [ ]:
sample_grid = [20, 50, 100, 250, 500, 1000]
ppl_results = {}
print("rung | final trace TV | evidence")
for rung in ladder:
    rows, evidence, exact = enumerate_traces(rung["prior"], rung["likelihood"], rung["obs"])
    errors = []
    for n in sample_grid:
        approx = importance_trace_inference(rung, n, rng)
        errors.append(total_variation(approx, exact))
    ppl_results[rung["name"]] = {"exact": exact, "errors": errors, "evidence": evidence}
    print(f"{rung['name']} | {errors[-1]:.4f} | {evidence:.3f}")

## Results visualization
The closing figure has target/sample panels on top and the requested error-vs-iteration or error-vs-sample curve below.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, rung in zip(axes[0], ladder):
    exact = ppl_results[rung["name"]]["exact"]
    approx = importance_trace_inference(rung, sample_grid[-1], rng)
    xs = np.arange(len(exact))
    ax.bar(xs - 0.15, exact, width=0.3)
    ax.bar(xs + 0.15, approx, width=0.3)
    ax.set_title(rung["name"], fontsize=8)
for ax, rung in zip(axes[1], ladder):
    ax.plot(sample_grid, ppl_results[rung["name"]]["errors"], marker="o")
    ax.set_xscale("log")
    ax.set_title("TV vs inference samples", fontsize=8)
fig.tight_layout()
plt.show()

## Pitfall on the hardest rung
The D5 pitfall is hiding impossible observations and confusing prior simulation with inference. If every trace has zero likelihood, conditioning collapses. The fix is an observation check plus conditioned enumeration or weighted trace sampling.

In [ ]:
d5 = ladder[-1]
impossible = dict(d5)
impossible["likelihood"] = np.column_stack([np.ones(len(d5["prior"])), np.zeros(len(d5["prior"]))])
try:
    enumerate_traces(impossible["prior"], impossible["likelihood"], 1)
except Exception as exc:
    print("caught impossible observation:", exc)
rows, evidence, exact = enumerate_traces(d5["prior"], d5["likelihood"], d5["obs"])
prior_simulation = d5["prior"]
conditioned = exact
print("prior simulation TV", round(float(total_variation(prior_simulation, conditioned)), 3))
print("conditioned posterior", conditioned)

## Evaluate it + Practice
- **Metric.** Total-variation distance between approximate and exact trace posteriors.
- **No-skill baseline.** Use prior simulation frequencies without conditioning.
- **Cheap sanity check.** Evidence must be positive and posterior trace weights must sum to 1.
- **Ablation.** Set all likelihoods for the observation to zero and verify the check fires.
- **Failure signals.** Zero evidence, prior-looking posterior, or one trace dominating only because proposal support is poor.

### Practice prompts
1. Add a second observed variable to D2 and enumerate traces.
2. Compare prior simulation and weighted trace inference on D3.
3. Create a soft impossible-observation check with a minimum evidence threshold.

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here